<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_PURGED_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Purged Walk-Forward v1 — audit de rigueur méthodologique (5/6)

**Rôle.** Nouveau notebook, indépendant, à lancer après `VIX_FINAL_FEATURES`. Ne teste pas une nouvelle feature ni un nouveau modèle : il **audite la méthodologie de validation walk-forward elle-même**, utilisée dans tous les notebooks précédents de ce projet.

**Le problème potentiel.** Le label à l'horizon h jours d'une ligne datée `d` est construit à partir du prix VIX en `d+h` (`vix.shift(-horizon)`). Pour les dernières `h-1` lignes de train juste avant la date de coupure d'un fold, ce label "regarde" une date qui tombe **dans la période de test** de ce même fold. Autrement dit, une poignée de lignes de train, proches de la frontière, encodent (via leur label) de l'information sur le futur immédiat qui appartient au test — un **look-ahead subtil**, distinct des fuites déjà corrigées dans ce projet (HMM lissé, Kalman non-causal, SMOTE avant fenêtrage) mais de la même famille.

**La méthode de correction : le "purging"** (López de Prado, *Advances in Financial Machine Learning*). On retire du train les lignes dont la fenêtre de label chevauche la période de test, avant d'entraîner.

**Ce que fait ce notebook.** Pour la grille (horizons × régimes × algos × folds), il entraîne et évalue **avec et sans purge**, à N=8 features (le point de fonctionnement établi) et sampler SMOTE fixe, pour isoler l'effet du seul changement de méthodologie. Il compare aussi le nombre de lignes réellement purgées (attendu ≈ h-1 en régime GLOBAL).

**Interprétation attendue** : si l'écart purgé vs non-purgé est négligeable (quelques lignes sur des centaines/milliers de train), les baselines déjà établies dans ce projet (GLOBAL RandomForest F1_dir≈0.610, TFT infirmé, etc.) restent valables telles quelles. Si l'écart est significatif, ces baselines devront être révisées avec purge systématique.


In [16]:
import subprocess, sys
pkgs = ['xgboost','lightgbm','catboost','shap','xlsxwriter','imbalanced-learn','pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [17]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.over_sampling import SMOTE

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_PURGED_CV'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizons': [1, 2, 3, 5, 7, 10],
    'regimes': ['GLOBAL', 'CALM', 'NORMAL', 'STRESS'],
    'algos': ['XGBoost', 'LightGBM', 'RandomForest', 'GradientBoosting', 'CatBoost'],
    'N': 8,                 # point de fonctionnement établi (GLOBAL RandomForest de référence)
    'sampler': 'SMOTE',      # fixé pour isoler l'effet du seul purging
    'n_wf_folds': 5,
    'min_train_frac': 0.40,  # doit matcher VIX_FINAL_FEATURES
    'shap_sample': 500,
    'pool_prefilter': 450,
    'min_train_rows': 100, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_purged_cv_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-purged-cv'

n_combos = (len(CONFIG['horizons']) * len(CONFIG['regimes']) * len(CONFIG['algos'])
            * CONFIG['n_wf_folds'] * 2)
print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | N={CONFIG['N']} sampler={CONFIG['sampler']} | "
      f"grille = {n_combos} lignes ({len(CONFIG['horizons'])}h × {len(CONFIG['regimes'])}reg × "
      f"{len(CONFIG['algos'])}algos × {CONFIG['n_wf_folds']}folds × 2 [purge/non-purge])")


VIX_PURGED_CV v1 | N=8 sampler=SMOTE | grille = 1200 lignes (6h × 4reg × 5algos × 5folds × 2 [purge/non-purge])


In [18]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features "
      f"(dont {len(meta['interaction_features'])} interactions) | source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


[SKIP] vix_final_features.parquet déjà présent localement
Dataset: (6908, 1300) | VIX=IDX_VIX | pool: 1102 features (dont 21 interactions) | source: 2000-01-03 → 2026-07-21
  Fold 1: train → 2010-08-16 | test 2010-08-17 → 2013-10-21
  Fold 2: train → 2013-10-21 | test 2013-10-22 → 2016-12-28
  Fold 3: train → 2016-12-28 | test 2016-12-29 → 2020-03-06
  Fold 4: train → 2020-03-06 | test 2020-03-09 → 2023-05-12
  Fold 5: train → 2023-05-12 | test 2023-05-15 → 2026-07-21


In [19]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    # Ensure y_true and y_pred are 1D arrays of integers
    y_true = np.asarray(y_true).ravel().astype(int)
    y_pred = np.asarray(y_pred).ravel().astype(int)

    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def get_clf(algo):
    if algo == 'XGBoost':
        return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=3, eval_metric='mlogloss',
                             objective='multi:softprob', random_state=SEED, n_jobs=-1, verbosity=0)
    if algo == 'LightGBM':
        return LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, num_leaves=31,
                              min_child_samples=10, subsample=0.8, class_weight='balanced',
                              random_state=SEED, verbose=-1, n_jobs=-1)
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1)
    if algo == 'GradientBoosting':
        return GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                          min_samples_leaf=10, subsample=0.8, random_state=SEED)
    if algo == 'CatBoost':
        return CatBoostClassifier(iterations=200, depth=6, learning_rate=0.05, loss_function='MultiClass',
                                  auto_class_weights='Balanced', random_state=SEED, verbose=False,
                                  allow_writing_files=False)
    raise ValueError(algo)

def get_samp(name):
    return {'SMOTE': SMOTE(random_state=SEED)}[name]

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:top_n]
    return list(keep[order])

print("Helpers OK (build_target, metrics, get_clf, get_samp, shap_rank)")

Helpers OK (build_target, metrics, get_clf, get_samp, shap_rank)


## Principe : purging (López de Prado)

En validation croisée classique, on suppose les observations indépendantes. En finance, ce n'est
jamais vrai : le **label** d'une ligne datée `d` à l'horizon `h` est une fonction du prix à `d+h`.
Si `d+h` tombe après la frontière train/test, cette ligne de train contient — via son label — de
l'information sur une date qui appartient au test. C'est un chevauchement (*overlap*), pas une
triche du modèle, mais un canal d'information résiduel que la validation croisée standard ignore.

**Purging** : avant d'entraîner, on retire du train toute observation dont la fenêtre d'évaluation
du label `[d, d+h]` chevauche la période de test. Concrètement ici, pour un horizon `h`, les
dernières observations de train dont la date-cible (`d+h` en jours de bourse) tombe à ou après la
date de coupure sont exclues — typiquement les `h-1` dernières lignes en régime GLOBAL (moins en
régime filtré, puisque les jours du régime ne sont pas contigus).

**Embargo** (le complément usuel du purging) empêche symétriquement le train de réutiliser des
observations situées juste après le test. Il est surtout utile en k-fold où le train encercle le
test des deux côtés. Ici, la validation est un **walk-forward à fenêtre expansive** strictement
chronologique (le train ne contient jamais de données postérieures au test) : il n'y a donc pas de
canal d'embargo à fermer côté "après" — seul le purging côté frontière train→test s'applique.


In [20]:
# ============================================================
# SYNCHRONISATION DE LA PROGRESSION (même principe que les autres notebooks)
# ============================================================
_PUSH_WORKDIR = "/content/_vix_purge_push"

def push_progress(label=''):
    if not GITHUB_TOKEN or not os.path.exists(RESULTS_CSV):
        return False
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0: return False
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        subprocess.run(["cp", RESULTS_CSV, f"{_PUSH_WORKDIR}/{RESULTS_CSV}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX Purged CV Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", RESULTS_CSV], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Progression Purged CV {label} — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        ok = push.returncode == 0
        if ok: print(f"  [CHECKPOINT PUSHÉ] {label}")
        return ok
    except Exception as e:
        print(f"  [WARN push checkpoint] {e}")
        return False

def pull_progress():
    if os.path.exists(RESULTS_CSV) or not GITHUB_TOKEN:
        return
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    exists = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", url, RESULTS_BRANCH],
                            capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"[INFO] Aucune progression antérieure sur '{RESULTS_BRANCH}'.")
        return
    workdir = "/content/_vix_purge_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", RESULTS_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode == 0 and os.path.exists(f"{workdir}/{RESULTS_CSV}"):
        subprocess.run(["cp", f"{workdir}/{RESULTS_CSV}", "."], check=True)
        print("[PULL OK] Progression Purged CV antérieure récupérée.")

pull_progress()


In [ ]:
# ============================================================
# COMPARAISON PURGÉ vs NON-PURGÉ, PAR (HORIZON, RÉGIME, ALGO, FOLD)
# ============================================================
KEY_COLS = ['horizon', 'regime', 'algo', 'fold', 'purge']

done_keys = set()
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    prev = pd.read_csv(RESULTS_CSV, usecols=KEY_COLS)
    done_keys = set(map(tuple, prev.values.tolist()))
    print(f"[REPRISE] {len(done_keys)} lignes déjà calculées.")

def save_row(row):
    header = not (os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=header, index=False)
    done_keys.add(tuple(row[c] for c in KEY_COLS))

vix_full = df_features[VIX_COL].ffill().bfill()
vix_dates = vix_full.index
n_vix = len(vix_dates)

t0 = time.time()
n_saved = 0
for h in CONFIG['horizons']:
    # date-cible (d + h jours de bourse) pour chaque date de la série complète — indépendant du fold
    fwd_pos = np.minimum(np.arange(n_vix) + h, n_vix - 1)
    fwd_date_of = pd.Series(vix_dates[fwd_pos], index=vix_dates)

    for reg in CONFIG['regimes']:
        for k in range(CONFIG['n_wf_folds']):
            cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
            cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
            target, reg_r, _ = build_target(df_features[VIX_COL], h, cut)
            idx = target.index
            base_tr_mask = np.asarray(idx < cut_date)
            te_mask = np.asarray((idx >= cut_date) & (idx <= nxt_date))
            if reg != 'GLOBAL':
                reg_al = reg_r.reindex(idx).fillna('NORMAL').values
                base_tr_mask = base_tr_mask & (reg_al == reg)
                te_mask = te_mask & (reg_al == reg)

            fwd = fwd_date_of.reindex(idx).values
            would_purge = fwd >= np.datetime64(cut_date)
            n_would_purge = int((base_tr_mask & would_purge).sum())

            y_te = target.values[te_mask].astype(int)
            if len(y_te) < CONFIG['min_test_rows']:
                continue

            X_pool = df_features[FEATURE_POOL].reindex(idx)
            X_te_raw = X_pool.values[te_mask]

            for algo in CONFIG['algos']:
                for purge in (False, True):
                    key = (h, reg, algo, k, purge)
                    if key in done_keys:
                        continue
                    tr_mask = base_tr_mask & (~would_purge if purge else True)
                    y_tr = target.values[tr_mask].astype(int)
                    if len(y_tr) < CONFIG['min_train_rows']:
                        continue
                    sc = RobustScaler()
                    X_tr = sc.fit_transform(np.nan_to_num(X_pool.values[tr_mask]))
                    X_te = sc.transform(np.nan_to_num(X_te_raw))
                    fidx = shap_rank(X_tr, y_tr, FEATURE_POOL, CONFIG['N'], CONFIG['pool_prefilter'])
                    try:
                        Xr, yr = get_samp(CONFIG['sampler']).fit_resample(X_tr[:, fidx], y_tr)
                    except Exception:
                        Xr, yr = X_tr[:, fidx], y_tr
                    clf = get_clf(algo); clf.fit(Xr, yr)
                    met = metrics(y_te, clf.predict(X_te[:, fidx]))
                    save_row({'horizon': h, 'regime': reg, 'algo': algo, 'fold': k, 'purge': purge,
                              'n_train': len(y_tr), 'n_test': len(y_te), 'n_would_purge': n_would_purge,
                              **met})
                    n_saved += 1
                    if n_saved % 200 == 0:
                        print(f"  ... {len(done_keys)}/{n_combos} lignes | {(time.time()-t0)/60:.1f}min")
                        push_progress(label=f"{len(done_keys)}/{n_combos}")

print(f"\n[PURGED CV] {len(done_keys)}/{n_combos} lignes calculées ({(time.time()-t0)/60:.1f}min)")
push_progress(label='fin de run')


[REPRISE] 89 lignes déjà calculées.
  ... 289/1200 lignes | 180.9min
  [CHECKPOINT PUSHÉ] 289/1200
  ... 489/1200 lignes | 360.9min
  [CHECKPOINT PUSHÉ] 489/1200


In [ ]:
# ============================================================
# SYNTHÈSE : EFFET DU PURGING SUR LES MÉTRIQUES
# ============================================================
df_p = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print(f"Progression: {len(df_p)}/{n_combos} ({len(df_p)/max(n_combos,1):.1%})")

if len(df_p):
    ok = df_p.dropna(subset=['F1_dir'])
    piv = ok.pivot_table(index=['horizon', 'regime', 'algo'], columns='purge',
                          values=['F1_dir', 'F1_UP_FORT', 'F1_DOWN_FORT', 'n_would_purge'],
                          aggfunc='mean')
    if (False in piv['F1_dir'].columns) and (True in piv['F1_dir'].columns):
        delta = pd.DataFrame({
            'F1_dir_non_purge': piv['F1_dir'][False].round(4),
            'F1_dir_purge': piv['F1_dir'][True].round(4),
            'delta_F1_dir': (piv['F1_dir'][True] - piv['F1_dir'][False]).round(4),
            'F1_UP_FORT_non_purge': piv['F1_UP_FORT'][False].round(4),
            'F1_UP_FORT_purge': piv['F1_UP_FORT'][True].round(4),
            'delta_F1_UP_FORT': (piv['F1_UP_FORT'][True] - piv['F1_UP_FORT'][False]).round(4),
            'n_would_purge_moyen': piv['n_would_purge'][False].round(1),
        }).reset_index().sort_values('delta_F1_dir')
        print("\n### Effet du purging par (horizon, régime, algo) — trié par delta F1_dir ###")
        print(delta.to_string(index=False))

        print(f"\nDelta F1_dir moyen (purgé - non-purgé) : {delta['delta_F1_dir'].mean():+.4f} "
              f"(std {delta['delta_F1_dir'].std():.4f})")
        print(f"Delta F1_UP_FORT moyen : {delta['delta_F1_UP_FORT'].mean():+.4f}")
        print(f"Lignes purgées en moyenne (régime GLOBAL, attendu ≈ h-1) : "
              f"{delta[delta['regime']=='GLOBAL']['n_would_purge_moyen'].mean():.1f}")

        if abs(delta['delta_F1_dir'].mean()) < 0.01 and abs(delta['delta_F1_UP_FORT'].mean()) < 0.02:
            print("\n[VERDICT] Effet du purging négligeable — les baselines walk-forward déjà "
                  "établies dans ce projet (GLOBAL RandomForest, TFT, etc.) ne sont PAS "
                  "matériellement affectées par ce look-ahead marginal aux frontières de fold.")
        else:
            print("\n[VERDICT] Effet du purging NON négligeable — les baselines établies devraient "
                  "être ré-auditées avec purging systématique avant d'être considérées définitives.")

        try:
            with pd.ExcelWriter('VIX_PURGED_CV_report.xlsx', engine='xlsxwriter') as w:
                delta.to_excel(w, 'Effet_purging', index=False)
                df_p.to_excel(w, 'Detail', index=False)
            print("\n[SAVE] VIX_PURGED_CV_report.xlsx (snapshot à date)")
        except Exception as e:
            print(f"[WARN Export] {e}")
    else:
        print("Pas encore assez de lignes des deux côtés (purge/non-purge) pour comparer.")
else:
    print("Aucun résultat pour l'instant.")
print(f"\n[NOTE] {RESULTS_CSV} contient le détail complet — le recharger pour reprendre.")


In [ ]:
# ============================================================
# PUSH FINAL DU RAPPORT (xlsx) EN PLUS DU CSV DE PROGRESSION
# ============================================================
def push_report_file():
    if not GITHUB_TOKEN or not os.path.exists('VIX_PURGED_CV_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["cp", "VIX_PURGED_CV_report.xlsx", f"{_PUSH_WORKDIR}/VIX_PURGED_CV_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_PURGED_CV_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport Purged CV agrégé — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_PURGED_CV_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_progress(label='rapport final')
push_report_file()
